# Assignment 7.1: Data Processing

For this assignment I'm using the United Nations spreadsheet on immigration to Canada, broken down by the country people come from. When I read it straight into pandas it looks one way (lots of code columns and a plain 0, 1, 2... index), and the assignment wants it to look another way: country names as the index, only the continent, region and development columns, one column for each year from 1980 to 2013, and a new Total column at the end.

**My plan:**
1. Read the correct sheet and skip the title rows at the top of it
2. Take a quick look at the data to know what I'm dealing with
3. Drop the columns I don't need
4. Rename the columns that have confusing names
5. Turn the year column labels into text
6. Make the country the index
7. Check the `Total` row that came with the spreadsheet and remove it
8. Add my own Total column
9. Compare the result with the picture in the assignment

## Step 1: Read the data

I only need pandas for this one.

In [1]:
# pandas does all the work in this notebook
import pandas as pd

The Excel file has more than one sheet, so before reading anything I want to see what the sheets are called.

In [2]:
# the spreadsheet should be in the same folder as this notebook
file_name = 'Assignment 7.1 data.xlsx'

# ExcelFile lets me peek at the workbook without loading a whole sheet yet
xls = pd.ExcelFile(file_name)
xls.sheet_names

['Regions by Citizenship', 'Canada by Citizenship']

There are two sheets. The first one, "Regions by Citizenship", only has totals for whole regions. The second one, "Canada by Citizenship", has one row per country and columns like `OdName`, `AREA` and `DEV`, which is what the "before" picture in the assignment shows. So I'll use the second sheet.

Before I use it, here is what happens if I just read it with the default settings:

In [3]:
# first attempt, no extra options, just to see what comes out
pd.read_excel(file_name, sheet_name='Canada by Citizenship').head(10)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36,Unnamed: 37,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,United Nations,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Population Division,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Department of Economic and Social Affairs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,International Migration Flows to and from Sele...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,POP/DB/MIG/Flow/Rev.2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


That is not usable. The first rows are the UN title, the copyright text and the citation, and the real column names are further down. Counting in the spreadsheet, the header row (`Type`, `Coverage`, `OdName`...) is at row 21, so I need to skip the first 20 rows.

In [4]:
# skiprows=20 jumps over the title and copyright text so the header row becomes the column names
df = pd.read_excel(file_name, sheet_name='Canada by Citizenship', skiprows=20)

# this should look like the first picture in the assignment
df.head()

,Type,Coverage,OdName,AREA,AreaName,REG,RegName,DEV,DevName,1980,...,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013
0,Immigrants,Foreigners,Afghanistan,935,Asia,5501,Southern Asia,902,Developing regions,16,...,2978,3436,3009,2652,2111,1746,1758,2203,2635,2004
1,Immigrants,Foreigners,Albania,908,Europe,925,Southern Europe,901,Developed regions,1,...,1450,1223,856,702,560,716,561,539,620,603
2,Immigrants,Foreigners,Algeria,903,Africa,912,Northern Africa,902,Developing regions,80,...,3616,3626,4807,3623,4005,5393,4752,4325,3774,4331
3,Immigrants,Foreigners,American Samoa,909,Oceania,957,Polynesia,902,Developing regions,0,...,0,0,1,0,0,0,0,0,0,0
4,Immigrants,Foreigners,Andorra,908,Europe,925,Southern Europe,901,Developed regions,0,...,0,0,1,1,0,0,0,0,1,1


This matches the "before" picture, so reading it worked.

## Step 2: Look at the data

I want to know how big it is and whether anything is missing before I start changing things.

In [5]:
# rows and columns
df.shape

(197, 43)

In [6]:
# the UN files sometimes use '..' for missing numbers, so I check both real NaNs and '..'
print('empty cells:', df.isnull().sum().sum())
print("cells with '..':", (df == '..').sum().sum())

empty cells: 0
cells with '..': 0


In [7]:
# what kinds of rows are there? Coverage says if the row is foreigners, citizens or both
df['Coverage'].value_counts()

Coverage
Foreigners    195
Citizens        1
Both            1
Name: count, dtype: int64

Nothing is missing, which makes my life easier. Almost all the rows are `Foreigners`, but there is also one `Citizens` row and one `Both` row. I'll look at those two again in step 7.

## Step 3: Drop the columns I don't need

Looking at the "after" picture, the new table only keeps the country, continent, region, development name and the years. `Type` is the same value in every row, `Coverage` I just checked, and `AREA`, `REG` and `DEV` are numeric codes for the names next to them (for example `AREA` 935 is just Asia), so they don't add anything.

In [8]:
# columns that are either constant or just codes for another column
cols_to_drop = ['Type', 'Coverage', 'AREA', 'REG', 'DEV']

# axis is not needed when I use columns=, and I don't use inplace so the cell is safe to re-run
df = df.drop(columns=cols_to_drop)
df.head()

,OdName,AreaName,RegName,DevName,1980,1981,1982,1983,1984,1985,...,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013
0,Afghanistan,Asia,Southern Asia,Developing regions,16,39,39,47,71,340,...,2978,3436,3009,2652,2111,1746,1758,2203,2635,2004
1,Albania,Europe,Southern Europe,Developed regions,1,0,0,0,0,0,...,1450,1223,856,702,560,716,561,539,620,603
2,Algeria,Africa,Northern Africa,Developing regions,80,67,71,69,63,44,...,3616,3626,4807,3623,4005,5393,4752,4325,3774,4331
3,American Samoa,Oceania,Polynesia,Developing regions,0,1,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
4,Andorra,Europe,Southern Europe,Developed regions,0,0,0,0,0,0,...,0,0,1,1,0,0,0,0,1,1


## Step 4: Rename the columns

The names the UN used are not very readable. I'm changing three of them to match the "after" picture. `DevName` stays the same because it is already called that in the picture.

In [9]:
# old name -> new name
df = df.rename(columns={
    'OdName': 'Country',      # OdName means origin/destination name, which is just the country here
    'AreaName': 'Continent',
    'RegName': 'Region',
})
df.columns.tolist()

['Country',
 'Continent',
 'Region',
 'DevName',
 1980,
 1981,
 1982,
 1983,
 1984,
 1985,
 1986,
 1987,
 1988,
 1989,
 1990,
 1991,
 1992,
 1993,
 1994,
 1995,
 1996,
 1997,
 1998,
 1999,
 2000,
 2001,
 2002,
 2003,
 2004,
 2005,
 2006,
 2007,
 2008,
 2009,
 2010,
 2011,
 2012,
 2013]

## Step 5: Make the year labels text

Right now the year column names are integers (1980, 1981, ...) while all the other names are strings. Mixing the two in one index is annoying, and I also want to add a column called `Total` that is a string. Converting the years to strings means every column name is the same type, and I can write `df['1980']` like any other column.

In [10]:
# turn every column name into a string
df.columns = list(map(str, df.columns))

# I'll need this list of year columns a few times later (1980 to 2013)
years = [str(year) for year in range(1980, 2014)]

# quick check: first three names, then the first and last year
print(df.columns.tolist()[:4])
print(years[0], years[-1])

['Country', 'Continent', 'Region', 'DevName']
1980 2013


## Step 6: Make Country the index

In the "after" picture the countries are the row labels, not a normal column. Setting it as the index also means I can look up a country directly with `.loc`.

In [11]:
df = df.set_index('Country')
df.head()

,Continent,Region,DevName,1980,1981,1982,1983,1984,1985,1986,...,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013
Country,,,,,,,,,,,,,,,,,,,,,
Afghanistan,Asia,Southern Asia,Developing regions,16,39,39,47,71,340,496,...,2978,3436,3009,2652,2111,1746,1758,2203,2635,2004
Albania,Europe,Southern Europe,Developed regions,1,0,0,0,0,0,1,...,1450,1223,856,702,560,716,561,539,620,603
Algeria,Africa,Northern Africa,Developing regions,80,67,71,69,63,44,69,...,3616,3626,4807,3623,4005,5393,4752,4325,3774,4331
American Samoa,Oceania,Polynesia,Developing regions,0,1,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
Andorra,Europe,Southern Europe,Developed regions,0,0,0,0,0,0,2,...,0,0,1,1,0,0,0,0,1,1


## Step 7: Check the `Total` row from the spreadsheet

When I checked the data earlier there was a `Both` row. I want to see exactly what it is, together with the `Unknown` and `Canada` rows, since they are not normal countries.

In [12]:
# look at the odd rows at the end and the Canada one
df.loc[['Canada', 'Unknown', 'Total'], ['Continent', 'Region', 'DevName', '1980', '2013']]

,Continent,Region,DevName,1980,2013
Country,,,,,
Canada,Northern America,Northern America,Developed regions,0,2
Unknown,World,World,World,44000,1484
Total,World,World,World,143137,259021


`Total` is not a country. It looks like the UN already added up all the countries for each year. If that is true, keeping it would be a problem: if somebody later summed a column they would count everything twice. Let me check if it really is the sum of the other rows.

In [13]:
# sum every year over all rows except Total, then compare with the Total row
sum_of_others = df.drop(index='Total')[years].sum()
total_row = df.loc['Total', years]

# True means all 34 years match exactly
bool((sum_of_others == total_row).all())

True

It matches for every year, so `Total` is only a summary row. I'm removing it. I'm keeping `Unknown` (people whose country of citizenship was not known) and `Canada` (people with Canadian citizenship coming into the country), since they are real categories in the data, and I'll mention them in the notes at the end.

In [14]:
# remove the summary row, everything else stays
df = df.drop(index='Total')
df.shape

(196, 37)

## Step 8: Add the Total column

The last thing in the convert picture is a `Total` column, which is the number of immigrants from each country over all the years 1980 to 2013. I add up across the year columns for each row. `axis=1` is what makes pandas sum along the row instead of down the column.

In [15]:
# sum of the 34 year columns for each country
df['Total'] = df[years].sum(axis=1)
df.head()

,Continent,Region,DevName,1980,1981,1982,1983,1984,1985,1986,...,2005,2006,2007,2008,2009,2010,2011,2012,2013,Total
Country,,,,,,,,,,,,,,,,,,,,,
Afghanistan,Asia,Southern Asia,Developing regions,16,39,39,47,71,340,496,...,3436,3009,2652,2111,1746,1758,2203,2635,2004,58639
Albania,Europe,Southern Europe,Developed regions,1,0,0,0,0,0,1,...,1223,856,702,560,716,561,539,620,603,15699
Algeria,Africa,Northern Africa,Developing regions,80,67,71,69,63,44,69,...,3626,4807,3623,4005,5393,4752,4325,3774,4331,69439
American Samoa,Oceania,Polynesia,Developing regions,0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,6
Andorra,Europe,Southern Europe,Developed regions,0,0,0,0,0,0,2,...,0,1,1,0,0,0,0,1,1,15


## Step 9: Compare with the assignment's "after" picture

In [16]:
# same shape as the picture: Country as index, then Continent, Region, DevName, years, Total
print('index name:', df.index.name)
print('columns   :', df.columns.tolist()[:3], '...', df.columns.tolist()[-3:])
print('shape     :', df.shape)

index name: Country
columns   : ['Continent', 'Region', 'DevName'] ... ['2012', '2013', 'Total']
shape     : (196, 38)


In [17]:
# the totals for the first five countries, to compare with the picture
df['Total'].head()

Country
Afghanistan       58639
Albania           15699
Algeria           69439
American Samoa        6
Andorra              15
Name: Total, dtype: int64

In the convert picture, Afghanistan has 58639, Albania 15699, Algeria 69439, American Samoa 6 and Andorra 15, and those are the same numbers I get. The first rows also have the same continent, region and development name as the picture, so the conversion matches the target.

## Summary and notes

What I did, in short: read the second sheet and skipped the 20 title rows, dropped the five columns that were constants or codes, renamed `OdName`, `AreaName` and `RegName`, converted the year labels to strings, set `Country` as the index, removed the UN's own `Total` row after checking it equals the sum of the other rows, and added a `Total` column by summing the years.